In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q05/annotated/checkpoints/post_cell_5.pickle

trying: ['STORAGE_OPTS']
me:  0
trying: ['orders']


me:  3
trying: ['supplier']
me:  11
trying: ['load_region']
me:  9
trying: ['load_customer']
me:  5
trying: ['orig_output']
me:  2
trying: ['region']
me:  9
trying: ['pd']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['load_nation']
me:  7
trying: ['customer']
me:  5
trying: ['nation']
me:  7
trying: ['load_supplier']
me:  11
trying: ['lineitem']


me:  1
trying: ['load_lineitem']
me:  1
trying: ['tpch_parent']
me:  0
trying: ['load_orders']
me:  3


Declaring variable STORAGE_OPTS
Declaring variable pd
Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable orig_output
Declaring variable orders
Declaring variable load_orders
Declaring variable load_customer
Declaring variable customer
Declaring variable load_nation
Declaring variable nation
Declaring variable load_region
Declaring variable region
Declaring variable supplier
Declaring variable load_supplier


In [4]:
%%RecordEvent
%%time
### cell 6 ###
# Filter on region and order year using GPU-friendly dt.year to avoid CPU-side Timestamp comparisons
rsel = region.R_NAME == "ASIA"
fregion = region[rsel]

# Use dt.year for a single GPU comparison instead of two CPU comparisons
osel = orders.O_ORDERDATE.dt.year == 1996
forders = orders[osel]

# Perform the sequence of merges as before (all GPU)
jn1 = fregion.merge(nation,   left_on="R_REGIONKEY", right_on="N_REGIONKEY")
jn2 = jn1.merge( customer,    left_on="N_NATIONKEY", right_on="C_NATIONKEY")
jn3 = jn2.merge( forders,     left_on="C_CUSTKEY",   right_on="O_CUSTKEY")
jn4 = jn3.merge( lineitem,    left_on="O_ORDERKEY",  right_on="L_ORDERKEY")
jn5 = supplier.merge(
    jn4,
    left_on=["S_SUPPKEY", "S_NATIONKEY"],
    right_on=["L_SUPPKEY", "N_NATIONKEY"]
)

# Compute revenue and aggregate
jn5["TMP"] = jn5.L_EXTENDEDPRICE * (1.0 - jn5.L_DISCOUNT)
total = (
    jn5.groupby("N_NAME", as_index=False, sort=False)["TMP"]
       .sum()
       .sort_values("TMP", ascending=False)
)

CPU times: user 113 ms, sys: 36.3 ms, total: 149 ms
Wall time: 156 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q05/rewritten/o4_mini_high_q5/checkpoints/post_cell_6_try_0.pickle

migration speed (bps): 797733903.4542315
---------------------------
variables to migrate:
pd 72
orders 48
load_orders 144
STORAGE_OPTS 64
jn2 48
load_customer 144
customer 48
load_lineitem 144
lineitem 48
fregion 48
rsel 48
total 48
tpch_parent 54
load_nation 144
nation 48
orig_output 16
region 48
load_region 144
jn1 48
osel 48
jn5 48
load_supplier 144
supplier 48
jn3 48
forders 48
jn4 48
DATA_ROOT 80
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q05/rewritten/o4_mini_high_q5/checkpoints/post_cell_6_try_0.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables ['customer']
Intermediate variables []
Future variables ['orders', 'lineitem']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables ['orders', 'customer', 'lineitem']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables ['region']
Future variables ['orders', 'nation', 'customer', 'lineitem']
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['customer', 'region', 'orders', 'lineitem', 'nation']
Modified dataframes
======= Cell 6 =======

In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q05/opt_cell_exec_info_6_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[6], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q05/annotated/checkpoints/post_cell_6.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
